# Predicting the sentiment of IMDb movie reviews

Step 1: Preprocessing the data for training

In [47]:
from re import split
import torch
from torchtext.datasets import IMDB

In [48]:
train_dataset_imdb = IMDB(split='train')
test_dataset_imdb = IMDB(split='test')

Split the training dataset into separate training and validation partitions.

In [49]:
from torch.utils.data.dataset import random_split
torch.manual_seed(1)
train_dataset, valid_dataset = random_split(list(train_dataset_imdb),[20000,5000])
len(train_dataset.dataset)

25000

 Find unique tokens (words)

In [50]:
from jinja2 import idtracking
import re
from collections import Counter, OrderedDict

def tokenizer(text):
    # Remove HTML tags
    text = re.sub(r'<[^>]*>', '', text)

    # Find emoticons
    emoticons = re.findall(
        r'(?::|;|=)(?:-)?(?:\)|\(|D|P)',
        text.lower()
    )

    # Remove non-word characters and convert to lowercase
    text = re.sub(r'[\W]+', ' ', text.lower())

    # Append emoticons (without hyphens)
    text = text + ' ' + ' '.join(emoticons).replace('-', '')

    # Split into individual tokens (words)
    tokenized = text.split()

    return tokenized

counter = Counter()
for label , line in train_dataset:
    tokens = tokenizer(line)
    counter.update(tokens)

print("Vocab size : ", len(counter))

Vocab size :  69006


encoding each unique token into integers

In [51]:
from torchtext.vocab import vocab

sorted_by_freq_tuples = sorted(counter.items(), key = lambda x : x[1], reverse=True)

ordered_dict = OrderedDict(sorted_by_freq_tuples)
vocab = vocab(ordered_dict)
vocab.insert_token("<pad>", 0)
vocab.insert_token("<unk>", 1)
vocab.set_default_index(1)

print([vocab[token] for token in ['this','is','an','example']])

[11, 7, 35, 457]


Transformation function for preparing data 

In [54]:
# Step 3-A: Define the functions for transformation
import torch.nn as nn

text_pipeline = lambda x: [vocab[token] for token in tokenizer(x)]
label_pipeline = lambda x: 1.0 if x == "pos" else 0.0


# Step 3-B: Wrap the encode and transformation function

def collate_batch(batch):
    label_list = []
    text_list = []
    lengths = []

    for _label, _text in batch:
        # Convert label to numeric value
        label_list.append(label_pipeline(_label))

        # Convert text to token IDs and then to a tensor
        processed_text = torch.tensor(
            text_pipeline(_text),
            dtype=torch.int64
        )

        text_list.append(processed_text)

        # Store the original sequence length
        lengths.append(processed_text.size(0))

    # Convert labels and lengths to tensors
    label_list = torch.tensor(label_list)
    lengths = torch.tensor(lengths)

    # Pad all sequences in the batch to the same length
    padded_text_list = nn.utils.rnn.pad_sequence(
        text_list,
        batch_first=True
    )

    return padded_text_list, label_list, lengths


# Take a small batch

from torch.utils.data import DataLoader

dataloader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=False,
    collate_fn=collate_batch
)

text_batch, label_batch, length_batch = next(iter(dataloader))

print(text_batch)
print(label_batch)
print(length_batch)

tensor([[   35,  1739,     7,   449,   721,     6,   301,     4,   787,     9,
             4,    18,    44,     2,  1705,  2460,   186,    25,     7,    24,
           100,  1874,  1739,    25,     7, 34414,  3568,  1103,  7517,   787,
             5,     2,  4991, 12401,    36,     7,   148,   111,   939,     6,
         11598,     2,   172,   135,    62,    25,  3199,  1602,     3,   928,
          1500,     9,     6,  4601,     2,   155,    36,    14,   274,     4,
         42944,     9,  4991,     3,    14, 10296,    34,  3568,     8,    51,
           148,    30,     2,    58,    16,    11,  1893,   125,     6,   420,
          1214,    27, 14542,   940,    11,     7,    29,   951,    18,    17,
         15994,   459,    34,  2480, 15211,  3713,     2,   840,  3200,     9,
          3568,    13,   107,     9,   175,    94,    25,    51, 10297,  1796,
            27,   712,    16,     2,   220,    17,     4,    54,   722,   238,
           395,     2,   787,    32,    27,  5236,  

Step 1.1: Split the data into train and test set

In [55]:
batch_size = 32

train_dl = DataLoader(train_dataset, batch_size=32, shuffle=True,collate_fn=collate_batch)
valid_dl = DataLoader(valid_dataset, batch_size=32, shuffle=False,collate_fn=collate_batch)
test_dl = DataLoader(test_dataset_imdb,batch_size=32, shuffle=False,collate_fn=collate_batch)

Step 2: Train the model

Creating the model

In [63]:
from PIL.ImageOps import pad
import torch.nn as nn

class RNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, rnn_hidden_size, fc_hidden_size):
        super().__init__()

        self.embeding = nn.Embedding(vocab_size,embed_dim,padding_idx=0)
        self.rnn = nn.LSTM(embed_dim,rnn_hidden_size,batch_first=True)

        self.fc1 = nn.Linear(rnn_hidden_size,fc_hidden_size)
        self.relu = nn.ReLU()

        self.fc2 = nn.Linear(fc_hidden_size,1)
        self.sigmoid = nn.Sigmoid()

    def forward(self,text,lengths):

        out = self.embeding(text)
        out = nn.utils.rnn.pack_padded_sequence( out, lengths.cpu().numpy(), enforce_sorted=False, batch_first=True )

        out , (hidden, cell ) = self.rnn(out)
        out = hidden[-1,:,:]

        out = self.fc1(out)
        out = self.relu(out)
        out = self.fc2(out)
        out = self.sigmoid(out)

        return out


vocab_size = len(vocab)
embed_dim = 20
rnn_hidden_size = 64
fc_hidden_size = 64

torch.manual_seed(1)

model = RNN(vocab_size,embed_dim,rnn_hidden_size,fc_hidden_size)
print(model)

RNN(
  (embeding): Embedding(69008, 20, padding_idx=0)
  (rnn): LSTM(20, 64, batch_first=True)
  (fc1): Linear(in_features=64, out_features=64, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=64, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)


Training the model

In [70]:
import torch.optim as optim

loss_fn = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

def train(dataloader):
    model.train()
    total_acc, total_loss = 0, 0
    for text_batch, label_batch, lengths in dataloader:
        optimizer.zero_grad()
        pred = model(text_batch, lengths)[:, 0]
        loss = loss_fn(pred, label_batch)
        loss.backward()
        optimizer.step()

        total_acc += (pred > 0.5).float().sum().item()
        total_loss += loss.item() * label_batch.size(0)

    return total_acc / len(dataloader.dataset), total_loss / len(dataloader.dataset)

In [ ]:
num_epochs = 10
torch.manual_seed(1)

for epoch in range(num_epochs):
    acc, loss = train(train_dl)
    acc_valid, loss_valid = evaluate(valid_dl)

    print(f"Epoch {epoch+1:02d} | Train Acc: {acc:.4f} Loss: {loss:.4f} | Valid Acc: {acc_valid:.4f} Loss: {loss_valid:.4f}")

Step 3: Evaluate the model on the test data

In [68]:
def evaluate(dataloader):
    model.eval()
    total_acc, total_loss = 0, 0
    with torch.no_grad():
        for text_batch, label_batch, lengths in dataloader:
            pred = model(text_batch, lengths)[:, 0]
            loss = loss_fn(pred, label_batch)

            total_acc += (pred > 0.5).float().sum().item()
            total_loss += loss.item() * label_batch.size(0)

    return total_acc / len(dataloader.dataset), total_loss / len(dataloader.dataset)

In [ ]:
num_epochs = 10

torch.manual_seed(1)

for epoch in range(num_epochs):

   
    print("Accur :  {acc} , Loss {loss}" )